# Notebook 2 — Entrenamiento, Ajuste de Hiperparámetros y Evaluación

## 1. Importación de librerías

In [ ]:
import warnings, os, joblib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.pipeline           import Pipeline
from sklearn.compose            import ColumnTransformer
from sklearn.impute             import SimpleImputer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.model_selection   import (GridSearchCV, StratifiedKFold,
                                        cross_val_score, train_test_split)
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (accuracy_score, precision_score,
                                        recall_score, f1_score, roc_auc_score,
                                        roc_curve, confusion_matrix,
                                        classification_report, ConfusionMatrixDisplay)

from xgboost   import XGBClassifier
from catboost  import CatBoostClassifier
from lightgbm  import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

SEED = 42

In [ ]:
# Cambiar el directorio de trabajo a la raíz del proyecto para que los archivos se guarden fuera de la carpeta Notebooks
import os
if os.getcwd().endswith('Notebooks'):
    os.chdir('..')
print(f"Directorio de trabajo actual: {os.getcwd()}")

## 2. Carga de datos y splits del Notebook 1

Se cargan los objetos exportados por el Notebook 1 para garantizar **reproducibilidad exacta** del mismo split y del mismo preprocesador.

In [ ]:
# ── Opción A: cargar desde Notebook 1 (recomendado) ─────────────────────────
try:
    X_train, X_test, y_train, y_test = joblib.load('data/train_test_splits.joblib')
    feature_lists   = joblib.load('data/feature_lists.joblib')
    numerical_features_all = feature_lists['numerical']
    categorical_features   = feature_lists['categorical']
    print('Splits cargados desde data/train_test_splits.joblib')

# ── Opción B: regenerar desde cero si no se ejecutó el Notebook 1 ───────────
except FileNotFoundError:
    print('Archivo no encontrado — regenerando splits desde la fuente...')
    URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    df  = pd.read_csv(URL)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)
    df.drop(columns=['customerID'], inplace=True)
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

    numerical_features_all = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
    categorical_features   = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod'
    ]
    X = df.drop(columns=['Churn'])
    y = df['Churn']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    print('Splits regenerados correctamente.')

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Churn en train: {y_train.mean():.3f}  |  Churn en test: {y_test.mean():.3f}')

## 3. Definición del preprocesador compartido

El mismo `ColumnTransformer` del Notebook 1, definido aquí para integrarse dentro de cada Pipeline de modelo. Se ajusta **dentro del GridSearchCV** en cada fold, evitando data leakage.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,    numerical_features_all),
    ('cat', categorical_transformer, categorical_features)
])

# Desbalance de clases: ratio negativa/positiva
neg, pos  = np.bincount(y_train)
scale_pos = neg / pos
print(f'scale_pos_weight para XGBoost / LightGBM: {scale_pos:.2f}')

## 4. Construcción de Pipelines y grillas de hiperparámetros

Cada Pipeline = preprocesador + clasificador. Los hiperparámetros siguen exactamente la tabla del enunciado (sección 5.12.5). Los nombres de los parámetros en la grilla usan el prefijo `clf__` para referirse a los parámetros del clasificador dentro del pipeline.

> **Nota sobre tiempos en Colab:** Para evitar tiempos de cómputo excesivos se usan grillas reducidas. Se puede ampliarlas comentando las versiones `_light` y descomentando las completas.

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=200, random_state=SEED,
        class_weight='balanced', n_jobs=-1
    ))
])

rf_param_grid = {
    'clf__max_depth':        [5, 10, None],
    'clf__min_samples_split': [2, 10],
    'clf__min_samples_leaf':  [1, 5],
    'clf__max_features':      ['sqrt', 'log2'],
    'clf__criterion':         ['gini', 'entropy']
}

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=200, random_state=SEED,
        scale_pos_weight=scale_pos,
        eval_metric='logloss', verbosity=0,
        use_label_encoder=False
    ))
])

xgb_param_grid = {
    'clf__max_depth':         [3, 6],
    'clf__learning_rate':     [0.05, 0.1],
    'clf__subsample':         [0.8, 1.0],
    'clf__colsample_bytree':  [0.8, 1.0],
    'clf__gamma':             [0, 1],
    'clf__min_child_weight':  [1, 5],
    'clf__reg_alpha':         [0, 0.1],
    'clf__reg_lambda':        [1, 5]
}

# ── CatBoost ─────────────────────────────────────────────────────────────────
cat_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', CatBoostClassifier(
        iterations=200, random_seed=SEED,
        auto_class_weights='Balanced',
        verbose=0
    ))
])

cat_param_grid = {
    'clf__depth':             [4, 6, 8],
    'clf__learning_rate':     [0.05, 0.1],
    'clf__l2_leaf_reg':       [1, 5],
    'clf__bagging_temperature': [0.5, 1.0],
    'clf__random_strength':   [1, 3],
    'clf__border_count':      [32, 128]
}

# ── LightGBM ─────────────────────────────────────────────────────────────────
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', LGBMClassifier(
        n_estimators=200, random_state=SEED,
        class_weight='balanced',
        verbose=-1,
        n_jobs=1          # ← CLAVE: evita explosión de hilos con GridSearch n_jobs=-1
    ))
])

lgbm_param_grid = {
    'clf__max_depth':         [6, 10],        # ← Eliminar -1; usar valores acotados
    'clf__learning_rate':     [0.05, 0.1],
    'clf__num_leaves':        [31, 63],
    'clf__min_child_samples': [20, 50],
    'clf__subsample':         [0.8, 1.0],
    'clf__colsample_bytree':  [0.8, 1.0],
    'clf__reg_alpha':         [0, 0.1],
    'clf__reg_lambda':        [1, 5],
}

print('Pipelines y grillas definidos:')
for name, grid in [('Random Forest', rf_param_grid), ('XGBoost', xgb_param_grid),
                   ('CatBoost', cat_param_grid), ('LightGBM', lgbm_param_grid)]:
    n_combos = 1
    for v in grid.values(): n_combos *= len(v)
    print(f'  {name}: {len(grid)} hiperparámetros, {n_combos:,} combinaciones × 5 folds')

## 5. Entrenamiento con GridSearchCV + Validación cruzada estratificada (5 folds)

Se usa `StratifiedKFold` con 5 particiones para mantener la proporción de clases en cada fold. La métrica de optimización es `roc_auc` por su robustez ante el desbalance de clases.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

experiments = [
    ('Random Forest', rf_pipeline,   rf_param_grid),
    ('XGBoost',       xgb_pipeline,  xgb_param_grid),
    ('CatBoost',      cat_pipeline,  cat_param_grid),
    ('LightGBM',      lgbm_pipeline, lgbm_param_grid),
]

best_estimators = {}
grid_results    = {}

for name, pipeline, param_grid in experiments:
    print(f'\n{"─"*55}')
    print(f'  Entrenando: {name}')
    print(f'{"─"*55}')

    gs = GridSearchCV(
        estimator  = pipeline,
        param_grid = param_grid,
        cv         = cv,
        scoring    = 'roc_auc',
        n_jobs     = -1,
        verbose    = 1,
        refit      = True          # Re-entrena sobre todo el train con mejores params
    )

    t0 = time.time()
    gs.fit(X_train, y_train)
    elapsed = time.time() - t0

    best_estimators[name] = gs.best_estimator_
    grid_results[name]    = gs

    print(f'  Mejor AUC (CV): {gs.best_score_:.4f}')
    print(f'  Tiempo: {elapsed:.1f}s')
    print(f'  Mejores parámetros:')
    for k, v in gs.best_params_.items():
        print(f'    {k}: {v}')


## 6. Evaluación sobre el conjunto de prueba

### 6.1 Función auxiliar de métricas

In [ ]:
def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    """Devuelve un dict con todas las métricas de evaluación."""
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]

    # Predicción con umbral ajustable
    y_pred_thr  = (y_prob >= threshold).astype(int)

    return {
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred_thr),
        'Precision': precision_score(y_test, y_pred_thr, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_thr, zero_division=0),
        'F1':        f1_score(y_test, y_pred_thr, zero_division=0),
        'AUC':       roc_auc_score(y_test, y_prob),
        'y_prob':    y_prob,
        'y_pred':    y_pred_thr,
        'CV_AUC':    grid_results[name].best_score_
    }

### 6.2 Tabla comparativa de métricas

In [ ]:
results_list = []
for name, model in best_estimators.items():
    results_list.append(evaluate_model(name, model, X_test, y_test))

# Separar métricas escalares de los arrays
metrics_df = pd.DataFrame([{
    k: v for k, v in r.items() if k not in ('y_prob', 'y_pred')
} for r in results_list])

metrics_df = metrics_df.set_index('Model').sort_values('AUC', ascending=False)

# Formatear para visualización
display_df = (metrics_df
              .drop(columns=['CV_AUC'])
              .style
              .format('{:.4f}')
              .background_gradient(cmap='YlGn', axis=0)
              .set_caption('Métricas en conjunto de prueba (threshold = 0.5)'))
display_df

In [ ]:
# AUC: CV vs Test — detectar overfitting
compare = metrics_df[['AUC', 'CV_AUC']].copy()
compare.columns = ['AUC Test', 'AUC CV (mean)']
compare['Diferencia'] = (compare['AUC Test'] - compare['AUC CV (mean)']).round(4)
print('Comparación AUC validación cruzada vs test:')
print(compare.to_string())

**Interpretación:**  
- Se espera que todos los modelos superen AUC 0.82 en test, dado que el dataset es relativamente limpio y las variables son discriminantes.
- Una diferencia CV↔Test menor a 0.02 indica buena generalización sin sobreajuste significativo.
- **Recall** es especialmente relevante en este contexto: identificar correctamente a los clientes que *sí* van a hacer churn es más crítico que minimizar falsos positivos, ya que una campaña de retención tiene coste menor que perder el cliente.
- Si Recall es bajo en todos los modelos, se puede bajar el umbral de decisión de 0.5 a ~0.3–0.4.

## 7. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

model_names = [r['Model'] for r in results_list]

for i, res in enumerate(results_list):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No Churn', 'Churn'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f"{res['Model']}\nAUC={res['AUC']:.4f} | F1={res['F1']:.4f}",
                      fontsize=10)
    axes[i].set_xlabel('Predicción')
    axes[i].set_ylabel('Valor real')

plt.suptitle('Matrices de Confusión — Conjunto de Prueba', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:**  
La matriz de confusión muestra cuatro cuadrantes:
- **TN (arriba-izq):** clientes que no hicieron churn y el modelo predijo correctamente.
- **FP (arriba-der):** falsos alarmas — se gastó retención en clientes que no se iban a ir.
- **FN (abajo-izq):** clientes que sí hicieron churn pero el modelo no detectó (**coste más alto**).
- **TP (abajo-der):** churn correctamente identificado.

En el contexto de telecomunicaciones, los **Falsos Negativos tienen mayor coste de negocio** que los Falsos Positivos. Un modelo con alto Recall es preferible aunque tenga Precision algo menor.

## 8. Curvas ROC

In [ ]:
palette = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

fig, ax = plt.subplots(figsize=(8, 6))

for i, res in enumerate(results_list):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=palette[i], linewidth=2,
            label=f"{res['Model']} (AUC = {res['AUC']:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Clasificador aleatorio')
ax.fill_between([0, 1], [0, 1], alpha=0.03, color='gray')

ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=11)
ax.set_title('Curvas ROC — Todos los modelos', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

**Interpretación:**  
La curva ROC representa el trade-off entre Sensibilidad (Recall) y Especificidad a distintos umbrales de decisión. Un AUC cercano a 1.0 indica que el modelo discrimina perfectamente entre las dos clases. La línea diagonal representa un clasificador sin información (AUC = 0.5). Todos los modelos deberían situarse considerablemente por encima de la diagonal. El modelo con mayor área bajo la curva es el más recomendable para producción si el criterio es discriminación general.

## 9. Reporte de clasificación detallado por modelo

In [ ]:
for res in results_list:
    print(f'\n{"═"*50}')
    print(f'  {res["Model"]}')
    print(f'{"═"*50}')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['No Churn', 'Churn'],
        digits=4
    ))

## 10. Importancia de características

### 10.1 Extracción de nombres de features tras OHE

In [ ]:
def get_feature_names(pipeline, num_cols, cat_cols):
    """Recupera los nombres de todas las features tras la transformación del pipeline."""
    prep = pipeline.named_steps['preprocessor']
    ohe  = prep.named_transformers_['cat']['onehot']
    ohe_names = ohe.get_feature_names_out(cat_cols).tolist()
    return num_cols + ohe_names

# Usar el pipeline de Random Forest (todos comparten el mismo preprocesador)
feature_names = get_feature_names(
    best_estimators['Random Forest'],
    numerical_features_all,
    categorical_features
)

print(f'Total de features: {len(feature_names)}')
print('Primeras 12:', feature_names[:12])

### 10.2 Top-10 variables más importantes por modelo

In [ ]:
def extract_importances(pipeline, feature_names):
    """Extrae feature_importances_ del clasificador dentro del pipeline."""
    clf = pipeline.named_steps['clf']
    imps = clf.feature_importances_
    fi = pd.DataFrame({'feature': feature_names, 'importance': imps})
    return fi.sort_values('importance', ascending=False).head(10)

palette_models = {
    'Random Forest': '#3498DB',
    'XGBoost':       '#E74C3C',
    'CatBoost':      '#2ECC71',
    'LightGBM':      '#F39C12'
}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for i, (name, model) in enumerate(best_estimators.items()):
    try:
        # Para modelos que usan el mismo preprocesador
        fn = get_feature_names(model, numerical_features_all, categorical_features)
    except Exception:
        fn = feature_names

    top10 = extract_importances(model, fn)

    # Limpiar nombres de features para mejor legibilidad
    top10['feature_clean'] = (top10['feature']
        .str.replace('cat__', '', regex=False)
        .str.replace('num__', '', regex=False)
        .str.replace('_', ' '))

    bars = axes[i].barh(
        top10['feature_clean'][::-1],
        top10['importance'][::-1],
        color=palette_models[name], alpha=0.85, edgecolor='white'
    )
    axes[i].set_title(f'{name} — Top 10 Features', fontweight='bold')
    axes[i].set_xlabel('Importancia relativa')
    axes[i].tick_params(axis='y', labelsize=9)

    # Añadir valores
    for bar, val in zip(bars, top10['importance'][::-1]):
        axes[i].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                     f'{val:.3f}', va='center', fontsize=8)

plt.suptitle('Top-10 Variables más Importantes por Modelo', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:**  
Se espera observar consistencia entre modelos: `tenure`, `MonthlyCharges`, `TotalCharges` y `Contract_One year` / `Contract_Two year` suelen ser los predictores más fuertes. La consistencia entre los cuatro modelos refuerza la validez de estas variables como señales reales de churn y no como artefactos de un algoritmo específico.

Variables como `gender` deberían aparecer con importancia baja o nula, confirmando lo observado en el EDA.

## 11. Comparación visual de métricas entre modelos

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
plot_df = metrics_df[metric_cols].reset_index()
plot_melted = plot_df.melt(id_vars='Model', value_vars=metric_cols,
                            var_name='Métrica', value_name='Valor')

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_melted, x='Métrica', y='Valor', hue='Model',
            palette=list(palette_models.values()), ax=ax, edgecolor='white')

ax.set_ylim(0.6, 1.0)
ax.set_title('Comparación de métricas entre modelos', fontsize=13)
ax.set_ylabel('Score')
ax.set_xlabel('')
ax.legend(title='Modelo', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))

# Línea de referencia
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

**Interpretación:**  
La gráfica permite comparar visualmente las cinco métricas para los cuatro modelos. La línea punteada en 0.80 sirve como referencia de un modelo aceptable para producción. Se espera que todos los modelos superen ese umbral en AUC, mientras que Recall puede estar por debajo de 0.80 dado el desbalance de clases con threshold=0.5.

## 12. Selección del mejor modelo y exportación

In [ ]:
best_model_name = metrics_df['AUC'].idxmax()
best_model      = best_estimators[best_model_name]
best_auc        = metrics_df.loc[best_model_name, 'AUC']

print(f'Mejor modelo: {best_model_name}')
print(f'AUC en test:  {best_auc:.4f}')
print(f'F1 en test:   {metrics_df.loc[best_model_name, "F1"]:.4f}')
print(f'Recall:       {metrics_df.loc[best_model_name, "Recall"]:.4f}')

In [ ]:
os.makedirs('app', exist_ok=True)

# Guardar el mejor pipeline completo (preprocesador + clasificador)
joblib.dump(best_model, 'app/model.joblib')

# Guardar también todos los modelos para el Notebook 3
os.makedirs('data', exist_ok=True)
joblib.dump(best_estimators, 'data/all_models.joblib')
joblib.dump(metrics_df,      'data/metrics_df.joblib')

size_kb = os.path.getsize('app/model.joblib') / 1024
print(f'Modelo exportado: app/model.joblib  ({size_kb:.1f} KB)')
print('Todos los modelos guardados en: data/all_models.joblib')
print('\nEste archivo (app/model.joblib) es el que usará la API FastAPI.')

## 13. Validación rápida del modelo exportado

In [ ]:
# Simular exactamente lo que hará la API FastAPI
loaded_model = joblib.load('app/model.joblib')

# Un registro de ejemplo (cliente de alto riesgo)
sample = pd.DataFrame([{
    'SeniorCitizen':    1,
    'tenure':           2,
    'MonthlyCharges':   99.9,
    'TotalCharges':     199.8,
    'gender':           'Male',
    'Partner':          'No',
    'Dependents':       'No',
    'PhoneService':     'Yes',
    'MultipleLines':    'Yes',
    'InternetService':  'Fiber optic',
    'OnlineSecurity':   'No',
    'OnlineBackup':     'No',
    'DeviceProtection': 'No',
    'TechSupport':      'No',
    'StreamingTV':      'Yes',
    'StreamingMovies':  'Yes',
    'Contract':         'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod':    'Electronic check'
}])

churn_prob  = loaded_model.predict_proba(sample)[0][1]
prediction  = 'Yes' if churn_prob > 0.5 else 'No'

print('=== Predicción de prueba (cliente de alto riesgo) ===')
print(f'Probabilidad de churn: {churn_prob:.4f} ({churn_prob*100:.1f}%)')
print(f'Predicción:            {prediction}')
print('\nEste formato coincide con la respuesta JSON de la API FastAPI.')

## 14. Resumen de resultados y conclusiones

### 14.1 Tabla final de mejores hiperparámetros

In [ ]:
print('Mejores hiperparámetros encontrados por GridSearchCV:\n')
for name, gs in grid_results.items():
    print(f'{name}:')
    for k, v in gs.best_params_.items():
        print(f'  {k.replace("clf__", "")}: {v}')
    print()

### 14.2 Conclusiones

| Modelo | Fortalezas observadas | Limitaciones |
|---|---|---|
| Random Forest | Robusto, estable, interpretable vía importancias | Más lento en búsqueda de hiperparámetros |
| XGBoost | Alta precisión, manejo nativo de desbalance | Requiere más ajuste de regularización |
| CatBoost | Excelente con categóricas, pocas iteraciones | Más pesado en memoria |
| LightGBM | Muy rápido, eficiente con datos tabulares | Puede sobreajustar con `num_leaves` alto |

**Decisión para producción:** Se exporta el modelo con mayor AUC en test. Para un entorno real, además del AUC se debería considerar el **tiempo de inferencia** y el **tamaño del modelo**, donde LightGBM suele ser más eficiente.

**Próximo paso:** El Notebook 3 aplicará LIME sobre los modelos entrenados para explicar predicciones individuales — especialmente útil para los equipos de retención que necesitan entender *por qué* un cliente en particular tiene alta probabilidad de churn.